# FLOW STATE | UAV Network Decision Analysis

**Purpose:** Explore how FLOW STATE's deterministic baseline selects a bandwidth policy under different simulated network conditions.

This notebook uses the project's actual `decisions.py` policy function. It does **not** represent real UAV telemetry or claim that Strands Decider 2B was called. The Strands path is a separate service integration and must be demonstrated only when that service is running and its response is verified.

**Repository:** https://github.com/suhailh3005-creator/flow-state

### What this notebook covers
1. Load the project code.
2. Run representative network scenarios through the deterministic policy.
3. Compare policy choices and visualize them.
4. Document the safety override and limitations.


## 1. Get the project source

Designed for Google Colab. If running locally, clone the repository and set `PROJECT_DIR` to your local folder.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_DIR = Path("/content/flow-state")

if not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/suhailh3005-creator/flow-state.git", str(PROJECT_DIR)],
        check=True
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project directory:", PROJECT_DIR)
print("Top-level files:")
for path in sorted(PROJECT_DIR.iterdir()):
    print(" •", path.name)


## 2. Load the baseline policy engine

The repository's `choose_policy()` function is a deterministic baseline, not an AI model. It maps bandwidth, latency, packet loss, and emergency status to one of four supported policies.

In [ ]:
from decisions import choose_policy, POLICIES
import pandas as pd
import matplotlib.pyplot as plt

print("Supported policies:", ", ".join(POLICIES.keys()))


## 3. Evaluate representative scenarios

The values below are **illustrative simulation inputs**, not field measurements. They are selected to exercise normal, degraded, severe, and emergency branches of the actual baseline function.

In [ ]:
scenarios = [
    {"scenario": "Normal network", "bandwidth_mbps": 8.0, "latency_ms": 40, "packet_loss_pct": 0.5, "emergency": False},
    {"scenario": "Moderate degradation", "bandwidth_mbps": 2.5, "latency_ms": 100, "packet_loss_pct": 1.0, "emergency": False},
    {"scenario": "High latency", "bandwidth_mbps": 5.0, "latency_ms": 220, "packet_loss_pct": 1.0, "emergency": False},
    {"scenario": "Severe packet loss", "bandwidth_mbps": 2.0, "latency_ms": 100, "packet_loss_pct": 12.0, "emergency": False},
    {"scenario": "Emergency mode", "bandwidth_mbps": 5.0, "latency_ms": 60, "packet_loss_pct": 1.0, "emergency": True},
]

results = []
for s in scenarios:
    decision = choose_policy(
        bandwidth_mbps=s["bandwidth_mbps"],
        latency_ms=s["latency_ms"],
        packet_loss_pct=s["packet_loss_pct"],
        emergency=s["emergency"],
    )
    results.append({
        **s,
        "selected_policy": decision["policy"],
        "reason": decision["reason"],
        "video_action": decision["actions"]["video"],
        "telemetry_action": decision["actions"]["telemetry"],
        "sensor_action": decision["actions"]["sensors"],
    })

results_df = pd.DataFrame(results)
results_df


## 4. Visualize the selected policies

This chart summarizes which policy the deterministic baseline selected for each scenario. It is a policy-selection view, not a performance benchmark.

In [ ]:
policy_counts = results_df["selected_policy"].value_counts().reindex(list(POLICIES.keys()), fill_value=0)

ax = policy_counts.plot(kind="bar", figsize=(9, 4))
ax.set_title("Baseline policy selections across illustrative scenarios")
ax.set_xlabel("Policy")
ax.set_ylabel("Number of scenarios")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


## 5. Inspect the action mapping

The action mapping below comes directly from `decisions.py`. In particular, emergency mode resolves to `REDUCE_VIDEO`, which reduces video demand and gives telemetry high priority.

In [ ]:
policy_actions_df = pd.DataFrame.from_dict(POLICIES, orient="index")
policy_actions_df.index.name = "policy"
policy_actions_df


## 6. Verify the emergency override

This assertion checks the behavior implemented by the deterministic baseline. It does not test or prove that the separate Strands Decider service is running.

In [ ]:
emergency_decision = choose_policy(
    bandwidth_mbps=8.0,
    latency_ms=30,
    packet_loss_pct=0.1,
    emergency=True,
)

assert emergency_decision["policy"] == "REDUCE_VIDEO"
assert emergency_decision["actions"]["telemetry"] == "high_priority"

print("Emergency policy:", emergency_decision["policy"])
print("Reason:", emergency_decision["reason"])
print("Actions:", emergency_decision["actions"])
print("Emergency override check: PASS")


## 7. Optional: run the project's offline evaluation script

If the repository includes `scripts/evaluate_offline.py`, this cell runs that project's own offline evaluation script. It skips cleanly if the script is not present. Offline evaluation does not require or prove a live Strands inference.

In [ ]:
evaluation_script = PROJECT_DIR / "scripts" / "evaluate_offline.py"

if evaluation_script.exists():
    print("Running the repository's offline evaluation script...")
    subprocess.run([sys.executable, str(evaluation_script)], cwd=PROJECT_DIR, check=True)
else:
    print("No scripts/evaluate_offline.py found in this repository revision; skipped.")


## 8. Interpretation and limitations

- `BALANCED`, `PROTECT_TELEMETRY`, `REDUCE_VIDEO`, and `DEFER_SENSORS` are the supported baseline policy IDs.
- The deterministic baseline is rule-based. It should not be described as a trained AI model.
- Network inputs and resulting performance metrics in this project are simulated.
- This notebook demonstrates the baseline policy function. It does not independently prove that Strands Decider 2B or Amazon Bedrock inference succeeded.
- To demonstrate the Strands path, run the actual service, select Strands Decider mode in the dashboard, and record the verified decision source and fallback status. Do not substitute fabricated outputs.

### Conclusion

FLOW STATE makes policy selection inspectable: given explicit network conditions, the baseline selects a supported policy and exposes the corresponding traffic actions. This provides a reproducible baseline against which a verified Decider 2B recommendation can be compared.
